## Resumé

Et logistik-driftsteam planlægger et randomiseret feltforsøg, der sammenligner tre chauffør-ruteplanlægningsstrategier (statisk baseline, dynamisk omdirigering, AI-optimeret) på tværs af tre depotregioner, med gennemsnitlig leveringsforsinkelse (minutter) som respons. PROC GLMPOWER tager et *eksemplarisk* datasæt med formodede celle-middelværdier og løser for det samlede antal chauffører, der er nødvendigt for at detektere strategieffekter ved 80% og 90% styrke, og kortlægger derefter, hvordan den opnåelige styrke aftager, efterhånden som rute-til-rute-variabiliteten vokser.

# Dimensionering af et chauffør-ruteplanlægningsforsøg med PROC GLMPOWER

## Resumé

Et logistik-driftsteam er ved at lancere et randomiseret feltforsøg, der sammenligner tre chauffør-ruteplanlægningsstrategier -- en **statisk** baseline, et **dynamisk** omdirigeringssystem og en **AI-optimeret** planlægger -- indsat på tværs af tre depotregioner (Northeast, Midwest, West). Responset er gennemsnitlig **leveringsforsinkelse i minutter**. Før teamet forpligter flådekapacitet til forsøget, skal det besvare: *hvor mange chauffører har vi brug for til pålideligt at kunne detektere den driftsforbedring, vi forventer?*

Denne notebook bruger **PROC GLMPOWER** til at udføre prospektiv styrke- og stikprøvestørrelsesanalyse for den generelle lineære model bag forsøget. Med udgangspunkt i et *eksemplarisk* datasæt med formodede celle-middelværdier (1) løser den for den samlede tilmelding, der kræves for at nå 80% og 90% styrke for de samlede strategi- og regionseffekter, (2) kortlægger, hvordan den opnåelige styrke forringes, efterhånden som rute-til-rute-variabiliteten stiger, og (3) producerer en styrkekurve til støtte for tilmeldingsbeslutningen.

> **Vigtigste konklusion:** Ved en plausibel residual-standardafvigelse på 8 minutter giver ca. **63 chauffører** 80% styrke, og **83 chauffører** giver 90% styrke til at detektere ruteplanlægningsstrategieffekter -- men hvis forsinkelsesvariabiliteten er tættere på 10 minutter, når selv 90 tilmeldte chauffører ikke op på 70% styrke, hvilket understreger værdien af stramme, veludstyrede ruter.

---
## 1. Byg det eksemplariske design

Det første trin koder forsøgslayoutet og teamets *formodede* gennemsnitlige forsinkelse for hver kombination af ruteplanlægningsstrategi x depotregion. Vi fastsætter et tilfældigt seed og tilføjer en ubetydelig støj, så middelværdierne ser realistiske ud, mens den balancerede 3x3-struktur bevares. `cell_n`-vægten på 1 i hver celle fortæller GLMPOWER, at designet er balanceret.

In [1]:
/* ================================================================
   Generér det eksemplariske datasæt med formodede middelforsinkelser.
   Én række pr. ruteplanlægningsstrategi x depotregion-designcelle.
   ================================================================ */
data routing_trial;
   CALL streaminit(20260531);
   LÆNGDE routing_strategy $8 depot_region $9;
   TABEL strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   TABEL region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   TABEL smean[3]     _temporary_ (18.0 14.5 11.0);   /* formodede strategimiddelværdier */
   TABEL radj[3]      _temporary_ (1.5  0.0 -1.0);    /* regionale justeringer (min)  */
   GØR i = 1 TIL 3;
      GØR j = 1 TIL 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         UDDATA;
      SLUT;
   SLUT;
   FJERN i j;
KØR;

PROCEDURE UDSKRIV data=routing_trial MÆRKAT noobs;
   VARIABEL routing_strategy depot_region mean_delay_min cell_n;
   MÆRKAT routing_strategy="Ruteplanlægningsstrategi" depot_region="Depotregion"
         mean_delay_min="Gennemsnitlig forsinkelse (min)" cell_n="Celle-n";
   TITEL "Eksemplariske cellemiddelværdier: Formodet leveringsforsinkelse (minutter)";
KØR;


                       Eksemplariske cellemiddelværdier: Formodet leveringsforsinkelse (minutter)                       

 Ruteplanlægningsstrategi  Depotregion  Gennemsnitlig forsinkelse (min)  Celle-n
Static                     Northeast                       19.687507651        1
Static                     Midwest                        17.9871067398        1
Static                     West                           16.8240210086        1
Dynamic                    Northeast                      15.9537535365        1
Dynamic                    Midwest                        14.4415169858        1
Dynamic                    West                           12.8636389493        1
AIOpt                      Northeast                      12.6143811724        1
AIOpt                      Midwest                         10.893885771        1
AIOpt                      West                             9.635351403        1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Stikprøvestørrelse for det samlede design

Med designet på plads beder vi GLMPOWER om at **løse for den samlede stikprøvestørrelse** (`NTOTAL = .`) ved 80% og 90% styrke. `MODEL`-sætningen angiver et to-vejs-layout med interaktion (`routing_strategy*depot_region`), `WEIGHT cell_n` erklærer de balancerede profilvægte, og `STDDEV = 8` er den antagne kvadratrod af middelkvadratfejlen for leveringsforsinkelsen. `NFRACTIONAL` lader proceduren rapportere nøjagtige fraktionelle antal før oprunding.

Vi forhåndsregistrerer også tre planlagte `CONTRAST`-sammenligninger -- AI-optimeret vs. statisk, dynamisk vs. statisk, og enhver omdirigering vs. statisk -- som dokumenterer de driftsmæssigt meningsfulde hypoteser, forsøget er bygget til at teste.

In [2]:
/* ================================================================
   Løs for det samlede antal chauffører, der kræves for at nå 80%
   og 90% styrke for ruteplanlægningsstrategi- og depotregionseffekterne.
   ================================================================ */
PROCEDURE glmpower data=routing_trial;
   KLASSE routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   VÆGT cell_n;
   CONTRAST "AI-optimeret vs. statisk"        routing_strategy -1  0  1;
   CONTRAST "Dynamisk vs. statisk"            routing_strategy -1  1  0;
   CONTRAST "Enhver omdirigering vs. statisk" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   MÆRKAT routing_strategy="Ruteplanlægningsstrategi" depot_region="Depotregion"
         mean_delay_min="Gennemsnitlig forsinkelse (min)";
   TITEL "Stikprøvestørrelse til at detektere ruteplanlægningsstrategieffekter på forsinkelse";
KØR;


                       Eksemplariske cellemiddelværdier: Formodet leveringsforsinkelse (minutter)                       

The GLMPOWER Procedure


                 Fixed Scenario Elements                 

Item                Value                                
------------------  -------------------------------------
Dependent Variable  Gennemsnitlig forsinkelse (min)      
Source              Ruteplanlægningsstrategi             
Source              Depotregion                          
Source              Ruteplanlægningsstrategi*Depotregion 

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Styrke-sensitivitet over for variabilitet og forsøgsstørrelse

Svaret på stikprøvestørrelsen afhænger af den antagne standardafvigelse for fejlleddet, som sjældent er kendt præcist på forhånd. Her vender vi spørgsmålet om: vi **fastsætter** flere kandidat-tilmeldingstotaler (`NTOTAL = 45 90 180`) og **løser for opnået styrke** (`POWER = .`) på tværs af et gitter af plausible standardafvigelser for forsinkelsen (6, 8 og 10 minutter). Den resulterende tabel er et sensitivitetskort -- den viser, hvor robust hver tilmeldingsplan er, hvis den virkelige rute-variabilitet viser sig at være højere end håbet.

In [3]:
/* ================================================================
   Sensitivitetsgitter: opnået styrke på tværs af kandidat-
   forsøgsstørrelser og plausible standardafvigelser for forsinkelsen.
   ================================================================ */
PROCEDURE glmpower data=routing_trial;
   KLASSE routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   VÆGT cell_n;
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   MÆRKAT routing_strategy="Ruteplanlægningsstrategi" depot_region="Depotregion"
         mean_delay_min="Gennemsnitlig forsinkelse (min)";
   TITEL "Opnået styrke på tværs af variabilitet og forsøgsstørrelsesscenarier";
KØR;


                       Eksemplariske cellemiddelværdier: Formodet leveringsforsinkelse (minutter)                       

The GLMPOWER Procedure


              Fixed Scenario Elements              

Item                Value                          
------------------  -------------------------------
Dependent Variable  Gennemsnitlig forsinkelse (min)
Source              Ruteplanlægningsstrategi       
Source              Depotregion                    

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Styrkekurve for tilmeldingsbeslutningen

Til sidst optegner vi en **styrkekurve** -- opnået styrke, efterhånden som den samlede tilmelding vokser fra 30 til 270 chauffører i trin på 30 -- for strategi-plus-region-modellen ved den forventede standardafvigelse på 8 minutter. Løsning af `POWER = .` på tværs af dette tilmeldingsgitter producerer kurven som en tabuleret `N Total` vs. `Power`-serie, så vi kan aflæse den tilmelding, hvor hvert af de konventionelle 80%- og 90%-mål krydses.

In [4]:
/* ================================================================
   Styrkekurve: opnået styrke vs. samlet antal tilmeldte chauffører,
   fejet fra 30 til 270 i trin på 30 ved den forventede standard-
   afvigelse på 8 min.
   ================================================================ */
PROCEDURE glmpower data=routing_trial;
   KLASSE routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   VÆGT cell_n;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   MÆRKAT routing_strategy="Ruteplanlægningsstrategi" depot_region="Depotregion"
         mean_delay_min="Gennemsnitlig forsinkelse (min)";
   TITEL "Styrkekurve: Opnået styrke vs. samlet antal tilmeldte chauffører";
KØR;


                       Eksemplariske cellemiddelværdier: Formodet leveringsforsinkelse (minutter)                       

The GLMPOWER Procedure


              Fixed Scenario Elements              

Item                Value                          
------------------  -------------------------------
Dependent Variable  Gennemsnitlig forsinkelse (min)
Source              Ruteplanlægningsstrategi       
Source              Depotregion                    

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Fortolkning og anbefaling

Analysen giver driftsteamet en forsvarlig tilmeldingsplan:

- **Baseline-dimensionering (afsnit 2).** Under antagelse af en residual-standardafvigelse på 8 minutter når den fulde to-vejs-model (strategi, region og deres interaktion) **80% styrke ved 63 chauffører** og **90% styrke ved 83 chauffører**. Ved oprunding for frafald ligger et tilmeldingsmål tæt på **90 chauffører** komfortabelt over 90%-grænsen.

- **Sensitivitet betyder noget (afsnit 3).** Styrken er højt følsom over for forsinkelsesvariabilitet. Ved 90 chauffører falder den opnåede styrke fra ~99% (SD=6) til ~87% (SD=8) til ~68% (SD=10). Et pilotforsøg med 45 chauffører er kun tilstrækkeligt, hvis variabiliteten er lav (~81% styrke ved SD=6), og er alvorligt underdimensioneret (~37%), hvis SD når 10. Den praktiske implikation: at investere i konsistente, veludstyrede ruter for at holde variabiliteten nede er lige så værdifuldt som at tilføje flere chauffører.

- **Styrkekurven (afsnit 4).** Optegnet for strategi-plus-region-modellen (uden interaktionsled, linsen brugt til sensitivitetsanalysen) stiger den opnåede styrke fra 0.37 ved 30 chauffører til 0.69 ved 60, 0.87 ved 90 og 0.95 ved 120, og flader ud over 0.99 ved 180. Aflæses kurven mod målene, nås 80% styrke ved omkring 77 chauffører og 90% ved omkring 99 -- en anelse højere end den fulde models dimensionering i afsnit 2, fordi fjernelse af interaktionsleddet spreder den samme effekt over færre modelfrihedsgrader.

**Anbefaling:** Tilmeld cirka **90 chauffører** (30 pr. ruteplanlægningsstrategi, balanceret på tværs af de tre depotregioner). Dette klarer 90% styrke for den fulde model under den forventede standardafvigelse på 8 minutter, holder ~87% styrke selv på den mere konservative reducerede-model-kurve, og holder forsøget lille nok til at kunne gennemføres inden for ét enkelt driftskvartal.

*Bemærk:* GLMPOWER analyserer det formodede *design*, ikke observerede resultater -- så troværdigheden af disse tal afhænger af de formodede middelværdier og standardafvigelsen. Teams bør revurdere dimensioneringen, når et kort pilotforsøg giver et empirisk estimat af leveringsforsinkelsens variabilitet.